In [0]:
%sql
USE workspace.bde;

In [0]:
%sql
-- Business Question 1

WITH monthly_aggregates AS (
    SELECT
        DATE_FORMAT(pickup_datetime,'yyyy-MM') AS year_month,   -- Month in format "YYYY-MM"
        
        -- (1) Total number of trips
        COUNT(*) AS total_trips,
        
        -- (4) Average number of passengers per trip
        ROUND(AVG(passenger_count), 2) AS avg_passengers,
        
        -- (5) Average amount paid per trip
        ROUND(AVG(total_amount), 2) AS avg_amount_per_trip,
        
        -- (6) Average amount paid per passenger
        ROUND(SUM(total_amount) / NULLIF(SUM(passenger_count),0), 2) AS avg_amount_per_passenger
    FROM taxi_with_locations_final
    GROUP BY DATE_FORMAT(pickup_datetime,'yyyy-MM')
),

-- Most trips by day of the week per month
day_counts AS (
    SELECT
        DATE_FORMAT(pickup_datetime,'yyyy-MM') AS year_month,
        DATE_FORMAT(pickup_datetime,'EEEE') AS day_of_week,  -- e.g., Monday, Tuesday
        COUNT(*) AS trips
    FROM taxi_with_locations_final
    GROUP BY DATE_FORMAT(pickup_datetime,'yyyy-MM'), DATE_FORMAT(pickup_datetime,'EEEE')
),
most_days AS (
    -- The day of week with the maximum trips per month
    SELECT
        year_month,
        FIRST(day_of_week) AS most_trips_day
    FROM (
        SELECT year_month, day_of_week, trips
        FROM day_counts
        ORDER BY year_month, trips DESC
    )
    GROUP BY year_month
),

-- Most trips by hour of the day per month
hour_counts AS (
    SELECT
        DATE_FORMAT(pickup_datetime,'yyyy-MM') AS year_month,
        HOUR(pickup_datetime) AS hour_of_day,  -- 0-23
        COUNT(*) AS trips
    FROM taxi_with_locations_final
    GROUP BY DATE_FORMAT(pickup_datetime,'yyyy-MM'), HOUR(pickup_datetime)
),
most_hours AS (
    -- The hour with the maximum trips per month
    SELECT
        year_month,
        FIRST(hour_of_day) AS most_trips_hour
    FROM (
        SELECT year_month, hour_of_day, trips
        FROM hour_counts
        ORDER BY year_month, trips DESC
    )
    GROUP BY year_month
)
SELECT
    m.year_month,                -- Month ("YYYY-MM")
    m.total_trips,               -- (1) Total trips
    d.most_trips_day,            -- (2) Day of week with most trips
    h.most_trips_hour,           -- (3) Hour of day with most trips
    m.avg_passengers,            -- (4) Avg passengers per trip
    m.avg_amount_per_trip,       -- (5) Avg amount per trip
    m.avg_amount_per_passenger   -- (6) Avg amount per passenger
FROM monthly_aggregates m
LEFT JOIN most_days d ON m.year_month = d.year_month
LEFT JOIN most_hours h ON m.year_month = h.year_month
ORDER BY m.year_month
LIMIT 20; 


year_month,total_trips,most_trips_day,most_trips_hour,avg_passengers,avg_amount_per_trip,avg_amount_per_passenger
2009-01,1226,Thursday,0,1.6,18.29,11.45
2010-09,204,Thursday,1,1.05,17.64,16.82
2011-01,2,Monday,23,1.0,12.8,12.8
2011-02,1,Tuesday,0,1.0,15.8,15.8
2012-09,3,Thursday,19,1.0,9.79,9.79
2014-01,14427669,Friday,19,1.69,14.37,8.48
2014-02,13915170,Saturday,19,1.68,14.5,8.62
2014-03,16543708,Saturday,19,1.68,14.66,8.74
2014-04,15767558,Wednesday,19,1.68,14.95,8.9
2014-05,16030454,Friday,19,1.68,15.52,9.24


In [0]:
%sql
--Business Question 2 
SELECT
    taxi_color,
    
    -- Trip duration in minutes
    ROUND(AVG((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0), 2) AS avg_duration_min,
    ROUND(PERCENTILE_APPROX((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0, 0.5), 2) AS median_duration_min,
    ROUND(MIN((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0), 2) AS min_duration_min,
    ROUND(MAX((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0), 2) AS max_duration_min,
    
    -- Trip distance in km
    ROUND(AVG(trip_distance), 2) AS avg_distance_km,
    ROUND(PERCENTILE_APPROX(trip_distance, 0.5), 2) AS median_distance_km,
    ROUND(MIN(trip_distance), 2) AS min_distance_km,
    ROUND(MAX(trip_distance), 2) AS max_distance_km,
    
    -- Speed in km/h = distance / duration_in_hours
    ROUND(AVG(trip_distance / ((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/3600.0)), 2) AS avg_speed_kmh,
    ROUND(PERCENTILE_APPROX(trip_distance / ((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/3600.0), 0.5), 2) AS median_speed_kmh,
    ROUND(MIN(trip_distance / ((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/3600.0)), 2) AS min_speed_kmh,
    ROUND(MAX(trip_distance / ((UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/3600.0)), 2) AS max_speed_kmh

FROM taxi_with_locations_final
WHERE taxi_color IN ('yellow', 'green')
  AND dropoff_datetime IS NOT NULL
  AND pickup_datetime IS NOT NULL
  AND UNIX_TIMESTAMP(dropoff_datetime) > UNIX_TIMESTAMP(pickup_datetime)  -- remove negative durations
GROUP BY taxi_color
ORDER BY taxi_color;



taxi_color,avg_duration_min,median_duration_min,min_duration_min,max_duration_min,avg_distance_km,median_distance_km,min_distance_km,max_distance_km,avg_speed_kmh,median_speed_kmh,min_speed_kmh,max_speed_kmh
green,13.55,10.50,1.00,359.98,2.96,1.93,0.02,310.99,12.68,11.52,1.0,70.0
yellow,14.38,11.25,1.00,360.00,3.04,1.7,0.02,380.2,11.74,10.3,1.0,70.0


In [0]:
%sql
-- Business Question 3

SELECT
    taxi_color,
    pickup_borough,
    dropoff_borough,
    DATE_FORMAT(pickup_datetime, 'yyyy-MM') AS year_month,   -- month
    DATE_FORMAT(pickup_datetime, 'EEEE') AS day_of_week,     -- day of week
    HOUR(pickup_datetime) AS hour_of_day,                    -- hour
    
    COUNT(*) AS total_trips,                                 -- total number of trips
    ROUND(AVG(trip_distance), 2) AS avg_distance_km,        -- average distance
    ROUND(AVG(total_amount), 2) AS avg_amount_per_trip,     -- average total_amount
    ROUND(SUM(total_amount), 2) AS total_amount_paid        -- total total_amount

FROM taxi_with_locations_final
WHERE taxi_color IN ('yellow','green')
  AND pickup_borough IS NOT NULL
  AND dropoff_borough IS NOT NULL
GROUP BY
    taxi_color,
    pickup_borough,
    dropoff_borough,
    DATE_FORMAT(pickup_datetime, 'yyyy-MM'),
    DATE_FORMAT(pickup_datetime, 'EEEE'),
    HOUR(pickup_datetime)
ORDER BY
    taxi_color,
    pickup_borough,
    dropoff_borough,
    year_month,
    day_of_week,
    hour_of_day;


taxi_color,pickup_borough,dropoff_borough,year_month,day_of_week,hour_of_day,total_trips,avg_distance_km,avg_amount_per_trip,total_amount_paid
green,Bronx,Bronx,2009-01,Thursday,0,4,0.5,2.53,10.1
green,Bronx,Bronx,2009-01,Thursday,3,1,0.51,5.8,5.8
green,Bronx,Bronx,2009-01,Thursday,9,1,0.73,6.3,6.3
green,Bronx,Bronx,2009-01,Thursday,10,1,1.43,9.8,9.8
green,Bronx,Bronx,2009-01,Thursday,15,1,4.42,18.3,18.3
green,Bronx,Bronx,2009-01,Thursday,16,1,6.05,26.8,26.8
green,Bronx,Bronx,2009-01,Thursday,17,1,3.31,19.8,19.8
green,Bronx,Bronx,2009-01,Thursday,18,1,3.04,15.8,15.8
green,Bronx,Bronx,2009-01,Thursday,23,1,1.56,10.8,10.8
green,Bronx,Bronx,2010-09,Thursday,0,5,2.28,12.26,61.32


In [0]:
%sql
-- Business Question 4

WITH revenue_by_pair AS (
    SELECT
        pickup_borough,
        dropoff_borough,
        ROUND(SUM(total_amount), 2) AS total_revenue
    FROM taxi_with_locations_final
    WHERE YEAR(pickup_datetime) = 2024
      AND pickup_borough IS NOT NULL
      AND dropoff_borough IS NOT NULL
    GROUP BY pickup_borough, dropoff_borough
),

-- Rank pairs by revenue
ranked_pairs AS (
    SELECT
        pickup_borough,
        dropoff_borough,
        total_revenue,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
    FROM revenue_by_pair
),

-- Compute total revenue across all pairs
total_revenue_cte AS (
    SELECT SUM(total_revenue) AS total_revenue_2024
    FROM revenue_by_pair
)

SELECT
    r.pickup_borough,
    r.dropoff_borough,
    r.total_revenue,
    ROUND(r.total_revenue / t.total_revenue_2024 * 100, 2) AS revenue_share_pct
FROM ranked_pairs r
CROSS JOIN total_revenue_cte t
WHERE r.revenue_rank <= 10
ORDER BY r.total_revenue DESC;


pickup_borough,dropoff_borough,total_revenue,revenue_share_pct
Manhattan,Manhattan,6.2943892533E8,61.3
Queens,Manhattan,1.687219933E8,16.43
Manhattan,Queens,6.803645259E7,6.63
Queens,Brooklyn,3.678260787E7,3.58
Manhattan,Brooklyn,3.124461275E7,3.04
Queens,Queens,2.979686175E7,2.9
Queens,Unknown,1.429371079E7,1.39
Manhattan,EWR,1.172087578E7,1.14
Manhattan,Unknown,6700299.92,0.65
Queens,Bronx,5426693.4,0.53


In [0]:
%sql
--Business Question 5

SELECT
    ROUND(
        100.0 * SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS pct_trips_with_tips
FROM taxi_with_locations_final
WHERE tip_amount IS NOT NULL;


pct_trips_with_tips
63.04


In [0]:
%sql
--Business Question 6

SELECT
    ROUND(
        100.0 * SUM(CASE WHEN tip_amount >= 15 THEN 1 ELSE 0 END) 
                 / COUNT(*),
        2
    ) AS pct_trips_tip_15_or_more
FROM taxi_with_locations_final
WHERE tip_amount > 0;


pct_trips_tip_15_or_more
0.82


In [0]:
%sql
-- Question 7

SELECT
    duration_bin,
    
    -- Average speed in km/h
    ROUND(AVG(trip_distance / (trip_duration_min / 60.0)), 2) AS avg_speed_kmh,
    
    -- Average distance per dollar
    ROUND(AVG(trip_distance / total_amount), 2) AS avg_distance_per_dollar,
    
    COUNT(*) AS num_trips
FROM (
    SELECT
        trip_distance,
        total_amount,
        -- Compute duration in minutes
        (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime)) / 60.0 AS trip_duration_min,
        
        -- Assign bins
        CASE
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 < 5 THEN 'Under 5 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 >= 5 
                 AND (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 < 10 THEN '5-10 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 >= 10 
                 AND (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 < 20 THEN '10-20 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 >= 20 
                 AND (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 < 30 THEN '20-30 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 >= 30 
                 AND (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60 < 60 THEN '30-60 Mins'
            ELSE '60+ Mins'
        END AS duration_bin
    FROM taxi_with_locations_final
    WHERE dropoff_datetime IS NOT NULL
      AND pickup_datetime IS NOT NULL
      AND UNIX_TIMESTAMP(dropoff_datetime) > UNIX_TIMESTAMP(pickup_datetime)
      AND trip_distance > 0
      AND total_amount > 0
) t
GROUP BY duration_bin
ORDER BY 
    CASE duration_bin
        WHEN 'Under 5 Mins' THEN 1
        WHEN '5-10 Mins' THEN 2
        WHEN '10-20 Mins' THEN 3
        WHEN '20-30 Mins' THEN 4
        WHEN '30-60 Mins' THEN 5
        ELSE 6
    END;


duration_bin,avg_speed_kmh,avg_distance_per_dollar,num_trips
Under 5 Mins,12.27,0.1,133881726
5-10 Mins,10.69,0.13,284499954
10-20 Mins,11.13,0.16,339066674
20-30 Mins,13.27,0.19,121951088
30-60 Mins,16.05,0.23,70803593
60+ Mins,14.21,0.4,8163619


In [0]:
%sql
-- Question 8
--Revenue contribution per bin (so we see where money is actually coming from).
WITH duration_binned AS (
    SELECT *,
        CASE
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0 < 5 THEN 'Under 5 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0 BETWEEN 5 AND 10 THEN '5-10 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0 BETWEEN 10 AND 20 THEN '10-20 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0 BETWEEN 20 AND 30 THEN '20-30 Mins'
            WHEN (UNIX_TIMESTAMP(dropoff_datetime) - UNIX_TIMESTAMP(pickup_datetime))/60.0 BETWEEN 30 AND 60 THEN '30-60 Mins'
            ELSE '60+ Mins'
        END AS duration_bin
    FROM taxi_with_locations_final
    WHERE dropoff_datetime IS NOT NULL
      AND pickup_datetime IS NOT NULL
      AND UNIX_TIMESTAMP(dropoff_datetime) > UNIX_TIMESTAMP(pickup_datetime)
)

-- Revenue + Tip behavior by duration bin
SELECT
    duration_bin,
    COUNT(*) AS num_trips,
    ROUND(SUM(total_amount), 2) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS avg_revenue_per_trip,
    ROUND(100.0 * SUM(CASE WHEN tip_amount > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_trips_with_tip,
    ROUND(AVG(tip_amount), 2) AS avg_tip,
    ROUND(AVG(tip_amount / NULLIF(total_amount,0)), 2) AS avg_tip_pct
FROM duration_binned
GROUP BY duration_bin
ORDER BY total_revenue DESC;


duration_bin,num_trips,total_revenue,avg_revenue_per_trip,pct_trips_with_tip,avg_tip,avg_tip_pct
10-20 Mins,337202495,5.4902811991E9,16.28,65.55,1.8,0.1
30-60 Mins,70552667,3.35816631848E9,47.6,66.09,5.31,0.1
20-30 Mins,121237855,3.34365732538E9,27.58,66.58,3.12,0.1
5-10 Mins,288509003,2.95088890316E9,10.23,61.83,1.12,0.1
Under 5 Mins,134389883,9.7403883318E8,7.25,54.84,0.77,0.09
60+ Mins,8164569,5.836900025E8,71.49,58.48,7.26,0.09
